In [37]:
# Assuming you have already installed prfpy_csenf

%load_ext autoreload
%autoreload 2
%matplotlib inline

import warnings
warnings.filterwarnings(action='ignore')
import sys
import os
opj = os.path.join
import time
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import cortex
import nibabel as nb
#
from prfpy_csenf.model import CSenFModel
from prfpy_csenf.stimulus import CSenFStimulus
from prfpy_csenf.fit import CSenFFitter
from prfpy_csenf.rf import * 
from prfpy_csenf.csenf_plot_functions import *

# Personal imports
sys.path.append("{}/../../../../analysis_code/utils".format(os.getcwd()))
from surface_utils import load_surface ,make_surface_image
from pycortex_utils import load_surface_pycortex, set_pycortex_config_file, draw_cortex, get_rois


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [32]:
# Set pycortex db and colormaps
cortex_dir = "{}/{}/derivatives/pp_data/cortex".format(main_dir, project_dir)
set_pycortex_config_file(cortex_dir)

In [2]:
# Settings
main_dir = '/Users/uriel/disks/meso_shared'
project_dir = 'nCSF'
subject = 'sub-01'
nCSF_ses = 'ses-01'
nCSF_task_name = 'nCSF'
format_ = 'fsnative'
TR = 1.6
n_jobs = 32

In [3]:
# Load events files 
events_dir = '{}/{}/{}/{}/func'.format(main_dir, project_dir, subject, nCSF_ses)
events_fn = '{}/{}_{}_task-{}_dir-PA_run-01_events.tsv'.format(events_dir, subject, nCSF_ses, nCSF_task_name)
events_df = pd.read_table(events_fn, sep="\t")



In [38]:
# Load data
data_dir = '{}/{}/derivatives/pp_data/{}/{}/func/fmriprep_dct_z-score_loo-avg'.format(main_dir, project_dir, subject, format_)
data_fn = '{}/{}_task-nCSF_hemi-L_fmriprep_dct_z-score_loo-avg-9_bold.func.gii'.format(data_dir, subject)

img, data = load_surface(data_fn)

In [31]:
data.shape

(214, 391393)

In [5]:
sf_minFreq = 0.05
sf_maxFreq = 16
sf_filtNum = 6 

minCont = 0.25
maxCont = 80
contNum = 12

sf_filtCenters = np.concatenate([np.round(np.logspace(np.log10(0.05), np.log10(16), sf_filtNum), 2), [0]])

contValues = np.concatenate([np.logspace(np.log10(minCont), np.log10(maxCont), contNum), [0]])

In [6]:
mapping_SF = {i+1: sf_filtCenters[i] for i in range(len(sf_filtCenters))}
mapping_MC = {i+1: contValues[i] for i in range(len(contValues))}

events_df['spatial_frequency'] = events_df['spatial_frequency'].replace(mapping_SF)
events_df['michelson_contrast'] = events_df['michelson_contrast'].replace(mapping_MC)

In [7]:
events_df['spatial_frequency']

0      0.0
1      0.0
2      0.0
3      0.0
4      0.0
      ... 
209    0.0
210    0.0
211    0.0
212    0.0
213    0.0
Name: spatial_frequency, Length: 214, dtype: float64

In [8]:
sfs_seq = np.array(events_df['spatial_frequency'])
con_seq = np.array(events_df['michelson_contrast'])

In [9]:
bounds = {
    'width_r'       : [0,10],          
    'SFp'           : [0, 20],
    'CSp'           : [0, 200] ,
    'width_l'       : [0.68, 0.68],     # we fix width_l in our model
    'crf_exp'       : [0, 10] ,
    'amp_1'         : [0, 6],       # Amplitude of TC
    'bold_baseline' : [-1,1] ,         # bold baseline (i.e., where is zero)
    'hrf_1'         : [1, 1],          # hrf derivative 
    'hrf_2'         : [0,0],           # hrf dispersion
}

grid_nr = 5 # number of steps in grid
width_r_grid        = np.linspace(bounds['width_r'][0], bounds['width_r'][1], grid_nr)     
SFp_grid            = np.linspace(bounds['SFp'][0], bounds['SFp'][1], grid_nr)     
CSp_grid            = np.linspace(bounds['CSp'][0], bounds['CSp'][1], grid_nr)
width_l_grid        = np.linspace(bounds['width_l'][0], bounds['width_l'][1], grid_nr)     
crf_exp_grid        = np.linspace(bounds['crf_exp'][0], bounds['crf_exp'][1], grid_nr)
hrf_1_grid = None
hrf_2_grid = None

grid_bounds = [bounds['amp_1']]
fixed_grid_baseline=False

bounds_list = [
    (bounds['width_r']),     # width_r
    (bounds['SFp']),     # SFp
    (bounds['CSp']),    # CSp
    (bounds['width_l']),     # width_l
    (bounds['crf_exp']),     # crf_exp
    (bounds['amp_1']),   # amp_1
    (bounds['bold_baseline']),      # baseline
    (bounds['hrf_1']),      # baseline
    (bounds['hrf_2']),      # baseline
]


In [10]:
width_l_grid

array([0.68, 0.68, 0.68, 0.68, 0.68])

In [11]:
# Create stim object
csenf_stim = CSenFStimulus(
    SF_seq  = sfs_seq, # np array, 1D 
    CON_seq = con_seq, # np array, 1D 
    TR      = TR,
    discrete_levels=True, # If discrete levels of SF and contrast. Default is true
    )

Number of timepoints: 214
Number of unique SF levels: 6, [ 0.05  0.16  0.5   1.59  5.05 16.  ]
Number of unique CON levels: 12, [ 0.25   0.422  0.714  1.205  2.037  3.441  5.813  9.82  16.591 28.029
 47.353 80.   ]


In [12]:
csenf_model = CSenFModel(stimulus=csenf_stim, 
                         hrf=[1,1,0], 
                         edge_type='CRF')

In [13]:
csenf_fitter = CSenFFitter(data=data.T, 
                           model=csenf_model, 
                           n_jobs=n_jobs)



In [14]:
width_r_grid.shape

(5,)

In [15]:
hrf_1_grid = np.array([0])
hrf_2_grid = np.array([0])
hrf_1_grid = np.zeros(len(width_r_grid))
hrf_2_grid = np.zeros(len(width_r_grid))

In [16]:
csenf_fitter.grid_fit(width_r_grid=width_r_grid, 
                      SFp_grid=SFp_grid, 
                      CSp_grid=CSp_grid, 
                      width_l_grid=width_l_grid, 
                      crf_exp_grid=crf_exp_grid, 
                      hrf_1_grid=hrf_1_grid, 
                      hrf_2_grid=hrf_2_grid, 
                      verbose=True,
                      fixed_grid_baseline=fixed_grid_baseline, 
                      grid_bounds=grid_bounds, 
                      n_batches=n_jobs)

Each batch contains approx. 12232 voxels.


[Parallel(n_jobs=32)]: Using backend LokyBackend with 32 concurrent workers.
/Users/uriel/softwares/prfpy_csenf/prfpy_csenf/fit.py:2101: RuntimeWarning: divide by zero encountered in divide
  slopes = (n_timepoints * np.dot(vox_data, predictions.T) - sumd *
/Users/uriel/softwares/prfpy_csenf/prfpy_csenf/fit.py:2101: RuntimeWarning: invalid value encountered in divide
  slopes = (n_timepoints * np.dot(vox_data, predictions.T) - sumd *
/Users/uriel/softwares/prfpy_csenf/prfpy_csenf/fit.py:2110: RuntimeWarning: invalid value encountered in multiply
  slopes[..., np.newaxis] *
/Users/uriel/softwares/prfpy_csenf/prfpy_csenf/fit.py:2101: RuntimeWarning: divide by zero encountered in divide
  slopes = (n_timepoints * np.dot(vox_data, predictions.T) - sumd *
/Users/uriel/softwares/prfpy_csenf/prfpy_csenf/fit.py:2101: RuntimeWarning: invalid value encountered in divide
  slopes = (n_timepoints * np.dot(vox_data, predictions.T) - sumd *
/Users/uriel/softwares/prfpy_csenf/prfpy_csenf/fit.py:2110:

In [ ]:
print(width_r_grid.shape)
print(SFp_grid.shape)
print(CSp_grid.shape)
print(width_l_grid.shape)
print(crf_exp_grid.shape)
print(hrf_1_grid.shape)
print(hrf_2_grid.shape)

In [18]:

# xtol, ftol: scipy optimize parameters, how quickly do you want the optimizer to stop looking for a solution
xtol = 0.0001
ftol = 0.0001

# rsq threshold. Throw away all fits which have a variance explained lower than:
rsq_threshold = 0.1 

In [20]:
# Start iterative fit
print('Starting iterative fit')
csenf_fitter.iterative_fit(
    rsq_threshold = rsq_threshold,
    verbose = False,
    bounds = bounds_list,
    # constraints = csf_constraints,
    # xtol=xtol,   
    # ftol=ftol,           
    )

Starting iterative fit


In [24]:
ncsf_fit = csenf_fitter.iterative_search_params

In [40]:
img_2 = make_surface_image(data=ncsf_fit.T, source_img=img)
nb.save(img_2,'/Users/uriel/Downloads/nCSF_params.func.gii')

In [48]:
ncsf_img_param, ncsf_data_param = load_surface('/Users/uriel/Downloads/nCSF_params.func.gii')

In [49]:
ncsf_data_param

array([[7.0852228e-02, 8.4051542e-02, 6.8751246e-02, ..., 0.0000000e+00,
        0.0000000e+00, 0.0000000e+00],
       [0.0000000e+00, 0.0000000e+00, 0.0000000e+00, ..., 0.0000000e+00,
        0.0000000e+00, 0.0000000e+00],
       [4.9920761e+01, 4.9847530e+01, 4.9724888e+01, ..., 2.0000000e+02,
        0.0000000e+00, 0.0000000e+00],
       ...,
       [1.0000000e+00, 1.0000000e+00, 1.0000000e+00, ..., 1.0000000e+00,
        0.0000000e+00, 0.0000000e+00],
       [0.0000000e+00, 0.0000000e+00, 0.0000000e+00, ..., 0.0000000e+00,
        0.0000000e+00, 0.0000000e+00],
       [4.4163680e-01, 3.8537574e-01, 3.8396597e-01, ..., 2.8783011e-01,
        0.0000000e+00, 0.0000000e+00]], shape=(10, 391393), dtype=float32)

In [44]:
ncsf_data_param

{}

In [50]:
vertex_data = cortex.Vertex(ncsf_data_param[:,1], subject)

cortex.quickshow(vertex_data, with_colorbar=False)

ValueError: Invalid number of vertices for subject (given 10, should be 229831 for left hem, 230015 for right hem, or 459846 for both)